# Bonsai-2 27B (ternary, W4A16-converted) on 1x CMP 170HX (SM80) — 1card, vLLM + DFlash2

| Metric | Value |
|---|---|
| Decode, c=1 (256-token cohort, ttft-excl.) | **155.7 tok/s** (wall 136.4) |
| Decode, c=1 (900-token cohort, ttft-excl.) | **251.6 tok/s** (wall 238.0) |
| Prefill | **1876 tok/s** at 6.6k prompt |
| TTFT, 10-token prompt | **226 ms** (client-observed) |
| Spec-decode acceptance | **4.17 accepted tokens/draft** (59.6% per position) |
| Best aggregate | 137.6 tok/s (flat to c=8, single-slot recipe) |

![served decode](../assets/charts/2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm-serve-decode.png)

```bash
# this notebook's weights do not exist on the Hub — they are CONVERTED from the
# ternary GGUF by the pipeline documented in section 3 (gguf -> bf16 -> W4A16).
# the ternary source:
hf download prism-ml/Ternary-Bonsai-2-27B-gguf Ternary-Bonsai-2-27B-F16.gguf --local-dir <weights>
```

Model: [prism-ml/Ternary-Bonsai-2-27B-gguf](https://huggingface.co/prism-ml/Ternary-Bonsai-2-27B-gguf) ·
companion ternary lane: [notebooks/2026-09-18-bonsai-2-27b-ternary-1card-llamacpp.ipynb](2026-09-18-bonsai-2-27b-ternary-1card-llamacpp.ipynb)


The 2026-09-18 ternary lane showed Bonsai-2 27B at **54.5 tok/s** on llama.cpp —
its 6.7 GB folded footprint is the point of the format, but llama.cpp has no
DFlash2 drafter for it. This notebook answers the follow-up: **put the same
Bonsai-2 weights through the exact mechanism of the 147.7 tok/s Qwen3.8-27B
recipe** (vLLM + DFlash2 k=7 speculative decoding, W4A16 main model).

To get there, the ternary weights had to leave the llama.cpp-only format:
the GGUF stores every folded weight in a Hadamard-rotated, v-grouped,
delta-normed layout that neither transformers nor vLLM can read. Section 3
documents the conversion (verified against the base model and executed
end-to-end), section 2 the measured results.

**Why the numbers beat the Qwen3.8-27B receipts slightly:** same recipe, three
deltas — vLLM 0.28.0 carries DFlash2 natively (the receipts used a 0.27.1
backport), the drafter's lookup-augmented drafting is on, and our W4A16 uses
int8 lm_head/embeddings. All measured on the same card class at the same
180 W cap. The ternary format's own advantage (6.7 GB resident vs 14 GB)
is **not** present here — that is the honest cost of the conversion.


In [1]:
# --- Status cell -------------------------------------------------------
# LIVE = False replays the committed receipts under results/2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm/receipts/.
import os

EXPERIMENT = "2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm"
RESULTS_DIR = os.path.join("..", "results", EXPERIMENT)
RECEIPTS = os.path.join(RESULTS_DIR, "receipts")
LIVE = False

print(f"experiment  : {EXPERIMENT}")
print(f"LIVE        : {LIVE} (receipts replayed; the run used vLLM on a loopback port)")


experiment  : 2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm
LIVE        : False (receipts replayed; the run used vLLM on a loopback port)


In [2]:
# --- Helpers ------------------------------------------------------------
import json
import os
import statistics as st

from IPython.display import Markdown, display


def receipt(name):
    with open(os.path.join(RECEIPTS, name)) as f:
        return json.load(f)


def jsonl(name):
    with open(os.path.join(RECEIPTS, name)) as f:
        return [json.loads(l) for l in f if l.strip()]


def render_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |",
             "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    display(Markdown("\n".join(lines)))


## 1. TL;DR

**Verdict: the mechanism transfers, and the converted Bonsai-2 weights are
fully competitive on it.** Single-stream decode lands at **155.7 tok/s**
(256-token cohort) and **251.6 tok/s** (900-token cohort — DFlash2's adaptive
draft depth stretches on the repetitive essay continuation), prefill at
**1876 tok/s** on a 6.6k prompt, TTFT **226 ms**. The base-calibrated DFlash2
drafter accepts **4.17 tokens per draft** (59.6% per position) against the
Bonsai-2 distribution with **zero recalibration** — the 98.2%-quality
retrain tracks the base model's distribution closely enough to draft for it.

Against the Qwen3.8-27B receipts on the same card class: decode256 is ~5%
above the 147.7 receipt and the 900-token cohort ~70% above it (adaptive
draft depth + vLLM 0.28), at the same 180 W cap. What is lost is the ternary
format's 6.7 GB residency — the W4A16 checkpoint is 18.6 GB (with int8
embed/lm_head/MTP) and its ceiling is bounded by that read, not by
speculation. The two lanes are complements, not alternatives: ternary for
footprint/energy (54.5 tok/s, 6.7 GB, 2.54 J/tok), W4A16+vLLM+DFlash2 for
throughput (155/252 tok/s, 18.6 GB).


In [3]:
meta = receipt("summary.json")
r = meta["results"]
key_metrics = [
    ("Decode 256-token cohort (tok/s, ttft-excl. / wall)", f"{r['decode256_ttft_excluded']} / {r['decode256_wall']}"),
    ("Decode 900-token cohort (tok/s, ttft-excl. / wall)", f"{r['decode900_ttft_excluded']} / {r['decode900_wall']}"),
    ("Prefill, ~6.6k-token prompt (tok/s)", r["prefill_6600"]),
    ("TTFT, 10-token prompt (ms, client-observed)", r["ttft_10tok_ms"]),
    ("Accepted tokens per draft (base-calibrated drafter, zero-shot)", r["acceptance"]["mean_accepted_per_draft"]),
    ("Per-position acceptance (7 draft slots)", ", ".join(f"{p/900:.2f}" for p in r["acceptance"]["accepted_per_pos"])),
    ("Aggregate at c=2/4/8 (tok/s, wall)", ", ".join(str(r["ladder_aggregate_wall"][c]) for c in ("2", "4", "8"))),
    ("Power during serving window (W, mean/peak)", "101 / 219 (180 W cap; transients above cap are sampled instantaneous draw)"),
    ("Peak core / memory temp (C)", "68 / 78"),
]
render_table(["Metric", "Value"], key_metrics)


| Metric | Value |
|---|---|
| Decode 256-token cohort (tok/s, ttft-excl. / wall) | 155.69 / 136.37 |
| Decode 900-token cohort (tok/s, ttft-excl. / wall) | 251.57 / 238.0 |
| Prefill, ~6.6k-token prompt (tok/s) | 1876.4 |
| TTFT, 10-token prompt (ms, client-observed) | 225.6 |
| Accepted tokens per draft (base-calibrated drafter, zero-shot) | 4.17 |
| Per-position acceptance (7 draft slots) | 0.88, 0.69, 0.61, 0.56, 0.50, 0.48, 0.46 |
| Aggregate at c=2/4/8 (tok/s, wall) | 137.6, 138.2, 136.9 |
| Power during serving window (W, mean/peak) | 101 / 219 (180 W cap; transients above cap are sampled instantaneous draw) |
| Peak core / memory temp (C) | 68 / 78 |

### Pins

From `receipts/summary.json`. The conversion chain is versioned by the
receipt itself: every step is deterministic given the GGUF and the manifest
inside it.


In [4]:
meta = receipt("summary.json")
pins = [
    ("Source weights", meta["weights"]["source"]),
    ("Conversion", "gguf -> bf16 (v-reorder + Hadamard unrotation + norm-delta +1 + graft) -> llmcompressor GPTQ W4A16"),
    ("bf16 probe", meta["weights"]["bf16_probe"]),
    ("Runtime", meta["runtime"]["vllm"]),
    ("Patches", meta["runtime"]["patches"]),
    ("FlashInfer", meta["runtime"]["flashinfer"]),
    ("GDN kernel deps", meta["runtime"]["extra"]),
    ("Drafter", meta["runtime"]["drafter"]),
    ("Serve flags", meta["runtime"]["flags"]),
    ("Hardware", "1x NVIDIA CMP 170HX (SM80, 64 GiB HBM2e), 180 W cap, forced airflow"),
    ("Driver / kernel", "610.43.03 / 6.8 (Ubuntu 22.04)"),
    ("Protocol", "greedy, streaming, usage-object counting, 1 warmup + 3 samples per cohort"),
]
render_table(["Pin", "Value"], pins)


| Pin | Value |
|---|---|
| Source weights | prism-ml/Ternary-Bonsai-2-27B-gguf F16 (Hadamard-folded, general.basename=folded) |
| Conversion | gguf -> bf16 (v-reorder + Hadamard unrotation + norm-delta +1 + graft) -> llmcompressor GPTQ W4A16 |
| bf16 probe | NLL 1.169 (ppl 3.22) on validation sentence; coherent generation |
| Runtime | 0.28.0 (DFlash2 native; dflash2-backport retired upstream) |
| Patches | syv-qwen38 patch series minus retired entries (33 applied) |
| FlashInfer | 0.6.16.post3 + cubin 0.6.13, JIT topk compiled (CUDA 13.0 toolkit nvcc) |
| GDN kernel deps | ['flash-linear-attention 0.5.2 (GDN prefill kernel)'] |
| Drafter | syvai/Qwen3.8-27B-DFlash2-W4A16 (1.19 GiB, base-calibrated, zero-shot for bonsai) |
| Serve flags | SPEC=dflash2 CTX=fast MAX_SEQS=1 DFLASH_TOKENS=7 GPU_UTIL=0.78 KV_MEM= |
| Hardware | 1x NVIDIA CMP 170HX (SM80, 64 GiB HBM2e), 180 W cap, forced airflow |
| Driver / kernel | 610.43.03 / 6.8 (Ubuntu 22.04) |
| Protocol | greedy, streaming, usage-object counting, 1 warmup + 3 samples per cohort |

### Protocol

Same shape as the 147.7-tok/s Qwen3.8-27B receipts: greedy (`temperature: 0`),
streaming with `stream_options.include_usage`, tokens counted from the final
usage object only, 1 warmup + 3 measured samples per cohort. Decode cohorts
use a 10-token prompt with `ignore_eos` at 256 and 900 completion tokens;
the prefill cohort uses a tokenizer-calibrated ~6.6k-token prompt with 11
output tokens and a **per-request nonce** so vLLM's prefix cache never serves
a repeated prefill. Acceptance counters are diffed from `/metrics`
(`vllm:spec_decode_num_accepted_tokens_per_pos_total` and friends) across the
decode window — an exact count, not a gauge.


## 2. Visible results

Every table is computed from the committed receipts.


### 2.1 Served decode, single stream

The 900-token cohort runs ~62% faster than the 256-token one: DFlash2's
adaptive draft depth (nmin 6, nmax 12) stretches the draft block as the
essay continuation stabilizes, and with `ignore_eos` the text is maximally
predictable. Both cohorts sit at or above the Qwen3.8-27B receipt on the
same card class.


In [5]:
pq = jsonl("serve-w4a16.jsonl")


def med(rows, tag, key):
    vals = [r[key] for r in rows if r["tag"].startswith(f"{tag}-sample") and r.get(key)]
    return st.median(vals)


d256 = [r["decode_tok_s"] for r in pq if r["tag"].startswith("decode256-sample")]
d900 = [r["decode_tok_s"] for r in pq if r["tag"].startswith("decode900-sample")]
rows = [
    ["decode 256 tok (ttft-excl.)", min(d256), st.median(d256), max(d256)],
    ["decode 900 tok (ttft-excl.)", min(d900), st.median(d900), max(d900)],
]
render_table(["Cohort", "min tok/s", "median tok/s", "max tok/s"], rows)

display(Markdown(f"![served decode](../assets/charts/2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm-serve-decode.png)"))


| Cohort | min tok/s | median tok/s | max tok/s |
|---|---|---|---|
| decode 256 tok (ttft-excl.) | 152.29 | 155.69 | 155.92 |
| decode 900 tok (ttft-excl.) | 251.23 | 251.57 | 253.67 |

![served decode](../assets/charts/2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm-serve-decode.png)

### 2.2 Speculative-decode acceptance, zero-shot drafter

The drafter (`syvai/Qwen3.8-27B-DFlash2-W4A16`) was calibrated on the BASE
model's hidden states — never on Bonsai-2. Acceptance decays smoothly from
0.88 (position 0) to 0.46 (position 6): the ternary retrain agrees with the
base distribution enough to draft for it, and every rejected draft is exact
(speculative decoding never changes the sampled distribution).


In [6]:
m = receipt("summary.json")
acc = m["results"]["acceptance"]
rows = [[f"pos {i}", p, f"{p/acc['drafts']:.2f}"] for i, p in enumerate(acc["accepted_per_pos"])]
rows.append(["**total**", acc["accepted_total"], f"**{acc['mean_accepted_per_draft']:.2f} per draft**"])
render_table(["Draft position", "accepted tokens", "rate"], rows)

display(Markdown(f"![acceptance](../assets/charts/2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm-acceptance.png)"))


| Draft position | accepted tokens | rate |
|---|---|---|
| pos 0 | 788 | 0.88 |
| pos 1 | 620 | 0.69 |
| pos 2 | 548 | 0.61 |
| pos 3 | 508 | 0.56 |
| pos 4 | 452 | 0.50 |
| pos 5 | 428 | 0.48 |
| pos 6 | 412 | 0.46 |
| **total** | 3756 | **4.17 per draft** |

![acceptance](../assets/charts/2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm-acceptance.png)

### 2.3 Prefill

Uncached prefill (per-request nonce defeats the prefix cache) at ~6.6k
tokens: 1876 tok/s median, within 4% of the Qwen3.8-27B receipt's 1955 on
the same card at the same power cap.


In [7]:
pf = [r for r in pq if r["tag"].startswith("prefill6k6-sample")]
render_table(
    ["Metric", "Value"],
    [
        ["Prompt tokens (usage-reported)", f"{pf[0]['prompt_tokens']:,}-{pf[-1]['prompt_tokens']:,} (nonce-varied)"],
        ["Prefill tok/s (min/median/max)", f"{min(r['prefill_tok_s'] for r in pf):.0f} / {st.median([r['prefill_tok_s'] for r in pf]):.0f} / {max(r['prefill_tok_s'] for r in pf):.0f}"],
    ],
)


| Metric | Value |
|---|---|
| Prompt tokens (usage-reported) | 6,612-6,615 (nonce-varied) |
| Prefill tok/s (min/median/max) | 1874 / 1876 / 1885 |

### 2.4 Concurrency

`MAX_SEQS=1` is the recipe of record for this lane, so the ladder measures
queueing: aggregate holds at the single-stream wall rate (~138 tok/s) while
per-stream rate divides by queue position. The 900-token cohort's adaptive
drafting does not engage here (256-token queue slots).


In [8]:
import re

lad = {}
for r in pq:
    m = re.fullmatch(r"ladder-c(\d+)-r(\d+)-SUMMARY", r["tag"])
    if m:
        lad.setdefault(int(m.group(1)), []).append(r["decode_tok_s"])
rows = [[c, st.median(v), round(st.median(v) / c, 1)] for c, v in sorted(lad.items())]
render_table(["concurrency", "aggregate tok/s (median of 3)", "per-stream tok/s"], rows)

display(Markdown(f"![ladder](../assets/charts/2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm-ladder.png)"))


| concurrency | aggregate tok/s (median of 3) | per-stream tok/s |
|---|---|---|
| 2 | 137.63 | 68.8 |
| 4 | 138.21 | 34.6 |
| 8 | 136.95 | 17.1 |

![ladder](../assets/charts/2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm-ladder.png)

### 2.5 Power and thermals

1 Hz telemetry across the serving window: mean 101 W, peak 219 W
(instantaneous transient above the average-based 180 W cap, same pattern as
earlier CMP 170HX receipts), 68 C core / 78 C memory peak — the 80/85 C
stop conditions were never approached.


In [9]:
import csv

ws, cores, mems = [], [], []
with open(os.path.join(RECEIPTS, "nvidia-vllm.csv")) as f:
    for row in csv.DictReader(f):
        ws.append(float(row[" power.draw [W]"].split()[0]))
        cores.append(int(row[" temperature.gpu"]))
        mems.append(int(row[" temperature.memory"]))
render_table(
    ["Window", "power min/mean/max W", "peak core C", "peak memory C", "samples"],
    [["serving", f"{min(ws):.0f}/{st.mean(ws):.0f}/{max(ws):.0f}", max(cores), max(mems), len(ws)]],
)

display(Markdown(f"![power](../assets/charts/2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm-power.png)"))


| Window | power min/mean/max W | peak core C | peak memory C | samples |
|---|---|---|---|---|
| serving | 41/101/219 | 68 | 78 | 272 |

![power](../assets/charts/2026-09-19-bonsai-2-27b-w4a16-dflash2-1card-vllm-power.png)

## 3. Reproduce — the conversion (the part that is not on the Hub)

The ternary GGUF stores three things that block a direct HF/vLLM load, and
each has an inverse that is verified against either the Prism runtime source
(`runtime.py`, C++ graph) or the base model:

1. **Hadamard-rotated basis.** Every folded weight is `W_stored = W_hf @ T`,
   `T = blockdiag(H_sylvest * diag(s_b) / sqrt(1024))` per 1024-block of the
   input dim, signs from the file's own `prism.hadamard.*` manifest
   (`sign_widths = [5120, 6144, 17408]`). Unfold: `W_hf = W_stored @ T`
   (T is orthogonal, so this is `W_stored @ T` with the manifest's own
   blocks — the runtime's activation-side transform is the same matrix).
2. **GDN v-layout.** `attn_qkv` (v segment), `attn_gate` (z), `ssm_alpha`,
   `ssm_beta`, `ssm_a`, `ssm_dt`, `ssm_conv1d` (v part) and `ssm_out`
   (columns) store the 48 v-heads **rep-major (3x16)**; HF wants
   **group-major (16x3)**. The permutation is `vperm` from the
   Bonsai-demo `runtime.py` `reorder()`, verified to 0.0 diff against the
   gen-1 `Ternary-Bonsai-27B-unpacked`/F16 pair.
3. **Delta norms.** `attn_norm`, `post_attention_norm`, `output_norm`,
   `attn_q_norm`, `attn_k_norm` are stored as `g - 1`; HF wants `g`
   (the runtime adds the 1 back inside its fused RMSNorm+rotate kernel).
4. **Grafts.** `model.visual.*` and `mtp.*` exist only in the base HF
   checkpoint (the ternary GGUF drops them); they are copied verbatim in
   bf16 — `mtp.*` is what DFlash2 drafts with.

Verified: NLL 1.169 on a validation sentence (base model: 1.518 through the
same harness), coherent greedy generation, structural index match to the
meta model. The conversion is deterministic and reproducible from the F16
GGUF.

**Then the club quant chain** (llmcompressor 0.13, GPTQ W4A16, 256 samples x
1024 tokens of open_platypus on 1 GPU): the syv `prepare/` scripts then
requant lm_head/embed/MTP to int8 and build the 40960-token draft head
(`mtp.draft_lm_head`, 213 MB). No prebuilt "fast variant" exists for
Bonsai-2 (the Hub one is base-model-only), so this lane runs the int8
variant — the receipts' fast variant is int4 and slightly faster.

**Serve** (vLLM 0.28.0, DFlash2 native; the syv patch series applies minus
the retired backport):

```bash
SPEC=dflash2 CTX=fast MAX_SEQS=1 DFLASH_TOKENS=7 GPU_UTIL=0.78 KV_MEM= \
  MODEL=<w4a16-dir> PORT=18020 bash single-user/start_qwen.sh
```

`GPU_UTIL=0.78` (not the recipe's 0.90) because the box co-hosts another
idle llama-server; with the card to yourself, 0.90 applies.


## 4. Appendix

<details>
<summary>Conversion debugging record, limitations, and open cells (click to expand)</summary>

### What the conversion debugging record shows (all measured)

The first four conversion attempts produced a *silently broken* model —
NLL pinned at ln(vocab) ~ 12.4 (uniform logits), coherent-looking structure,
correct shapes. The record is preserved because it is the interesting part:

- **Double-normalized Hadamard** (1/32 applied twice): all folded weights
  scaled to ~nothing; NLL ~12.1-12.4. RMSNorm absorbs a *uniform* weight
  scale, so every probe looked identical — scale bugs are invisible to NLL.
- **Transpose bug in the einsum** (`obk,bjk->obj` is `W @ T^T`, not `W @ T`):
  the transform was silently the inverse of the intended one. Two "fixes"
  changed nothing because both applied T^-1.
- Diagnosis that actually worked: (a) column-norm correlation of the
  converted lm_head against the base model's (0.9991 for the *unrotated*
  read — proving the F16 GGUF stores natural-basis head rows); (b)
  class-by-class weight swap against the base model (embed/head ✓, full-attn
  ✓, GDN ✓, FFN ✓ — isolating the norm delta); (c) per-layer hidden-magnitude
  hooks (embedding absmax 2.5e-05 = the smoking gun).

### Honesty notes

- The 900-token cohort's 252 tok/s is the adaptive draft depth engaging on
  `ignore_eos` essay text — maximally predictable content. Chat-shaped
  decoding lands nearer the 256-token cohort's 155 tok/s.
- The drafter is zero-shot (base-calibrated). Recalibrating it on Bonsai-2
  hidden states (the syv `drafter/capture_dflash2.py` path) is the obvious
  next lever for acceptance above 0.6.
- This lane runs the **int8** lm_head/embed/MTP variant (no Bonsai-2 fast
  variant exists on the Hub); the receipts' fast variant was int4.
- The 219 W peak is instantaneous sampled draw above the average-based
  180 W cap; the cap was read back on every telemetry sample.
- GPU 0 co-hosted an idle llama-server during the run (hence GPU_UTIL 0.78);
  idle, it does not affect decode.

### Untested

- vLLM concurrency beyond the queueing ladder (the recipe pins MAX_SEQS=1).
- Recalibrated drafter (see above).
- Vision tower (text-only benchmark).
- Quality benchmarks: Bonsai-2's published 84.78 thinking-mode average is
  community-reported; the W4A16 conversion adds GPTQ error on top (llmcompressor
  reported per-layer relative errors ~0.6-0.8% during quantization).

</details>


In [10]:
# --- Try your own prompt -------------------------------------------------
# Edit PROMPT and re-run against a live server (LIVE harness; no receipts touched).
import json
import os
import urllib.request

PROMPT = "Explain quantum computing in simple terms."  # keep max_tokens bounded: the model thinks (reasoning) before answering
BASE = os.environ.get("BENCH_ENDPOINT_URL", "http://127.0.0.1:18020")

payload = {
    "messages": [{"role": "user", "content": PROMPT}],
    "temperature": 0,
    "max_tokens": 256,
}
req = urllib.request.Request(
    BASE + "/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req, timeout=600) as resp:
    out = json.loads(resp.read())
print("content :", out["choices"][0]["message"]["content"][:400])
print("usage   :", json.dumps(out["usage"]))


content : 

Quantum computing is a way of computing that uses the rules of quantum physics to solve certain problems faster than ordinary computers.

A normal computer uses bits: each bit is either 0 or 1. A quantum computer uses **qubits**, which can be in a blend of 0 and 1 at the same time. This blend is called **superposition**.

Imagine a normal computer is like a light switch: it is either on or off. 
usage   : {"prompt_tokens": 60, "total_tokens": 316, "completion_tokens": 256, "prompt_tokens_details": {"cached_tokens": 0, "created_cache_tokens": 0, "multimodal_tokens": null}, "completion_tokens_details": {"reasoning_tokens": 85}}
